# Finance Business Questions

This notebook answers the four Finance questions using Gold Delta tables. The reporting metric is Net Revenue: extended price after discount, excluding tax.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src.config import GOLD_PREFIX, project_table

In [0]:
customer_revenue = spark.table(project_table(GOLD_PREFIX, "customer_revenue"))
yearly_revenue = spark.table(project_table(GOLD_PREFIX, "yearly_revenue"))
supplier_performance = spark.table(project_table(GOLD_PREFIX, "supplier_performance"))
segment_yearly_order_value = spark.table(
    project_table(GOLD_PREFIX, "segment_yearly_order_value")
)

## Question 1 - Top 10 customers by net revenue

The share is calculated against total Gold customer revenue. Customers with no orders remain in Gold with zero revenue, but do not affect the top-ten ranking.

In [0]:
total_revenue = customer_revenue.agg(
    F.sum("total_net_revenue").alias("total_net_revenue")
).first()["total_net_revenue"]

top_10_customers = (
    customer_revenue
    .orderBy(F.desc("total_net_revenue"), F.asc("c_custkey"))
    .limit(10)
    .withColumn("total_revenue", F.lit(total_revenue))
    .withColumn("revenue_share", F.col("total_net_revenue") / F.col("total_revenue"))
)
display(top_10_customers)

top_10_share = top_10_customers.agg(F.sum("revenue_share").alias("top_10_revenue_share"))
display(top_10_share)

The top ten customers together contribute approximately 0.00614% of total Net Revenue. The largest individual customer is `Customer#000721180` from the BUILDING segment, with approximately $6.88 million in Net Revenue. The very small combined share indicates that revenue is broadly distributed across the customer base rather than concentrated in a few accounts.

## Question 2 - 1997 revenue in three ways

Gross Revenue uses the unadjusted extended price. Net of Discount removes customer discounts. Net of Discount and Tax additionally includes tax. The recommended business metric is Net Revenue (Net of Discount): it reflects sales revenue after discounts while excluding tax collected on behalf of the government.

In [0]:
revenue_1997 = yearly_revenue.filter(F.col("year") == 1997)
display(revenue_1997)

display(
    revenue_1997.select(
        (F.col("gross_revenue") - F.col("net_revenue_discount")).alias("gross_minus_discount"),
        (F.col("net_revenue_discount") - F.col("net_revenue_discount_tax")).alias("discount_minus_tax_inclusive"),
    )
)

For 1997, Gross Revenue was approximately $173.95 billion, while Net Revenue after discount was approximately $165.25 billion. The difference of about $8.70 billion represents discounts. Tax-inclusive revenue was approximately $171.86 billion, around $6.61 billion above Net Revenue because it includes tax collected from customers. Net Revenue after discount is therefore the most appropriate headline business metric: it reflects sales after discounts without treating collected tax as retained company revenue.

## Question 3 - Average order value by market segment, year over year

The table contains one row per year and market segment. The window rank identifies the highest average order value in each year.

In [0]:
segment_rank_window = Window.partitionBy("year").orderBy(F.desc("avg_order_value"))
segment_aov_yoy = (
    segment_yearly_order_value
    .withColumn("segment_rank", F.dense_rank().over(segment_rank_window))
    .orderBy("year", "segment_rank", "c_mktsegment")
)
display(segment_aov_yoy)
display(segment_aov_yoy.filter(F.col("segment_rank") == 1))

The segment with the highest average order value changes over time: FURNITURE leads in 1992 and 1996, HOUSEHOLD in 1993, MACHINERY in 1994, 1997, and 1998, and AUTOMOBILE in 1995. The leading values are close to one another, so the result suggests small year-to-year differences rather than a permanently dominant segment. The 1998 result should be treated cautiously because 1998 is a partial year.

## Question 4 - Top 5 suppliers by net revenue

Average discount is the line-item average discount for each supplier's fulfilled orders.

In [0]:
top_5_suppliers = (
    supplier_performance
    .orderBy(F.desc("supplier_net_revenue"), F.asc("s_suppkey"))
    .limit(5)
)
display(top_5_suppliers)

The top five suppliers by Net Revenue are suppliers 7491, 24503, 41492, 22496, and 24983. Their individual revenues range from approximately $28.94 million to $29.24 million, and their average discounts are close to 5%. The narrow revenue range suggests that the leading suppliers contribute at broadly similar levels rather than one supplier dominating the group.

## Reporting conclusion

Use `net_revenue_discount` as the primary Finance reporting metric. Gross extended price is a catalogue-volume measure, while tax-inclusive revenue contains amounts collected for the government and therefore does not represent the company's retained sales revenue.